In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob


data_path = "./data/"
csv_files = glob.glob(data_path + "*.csv")

dataframe = {}

for file in csv_files:
    name = os.path.basename(file).replace(".csv","")
    dataframe[name] = pd.read_csv(file)

In [2]:
orders = dataframe["olist_orders_dataset"]

In [3]:
#统计不同订单状态的数量
status_count = orders["order_status"].value_counts()
#统计不同订单状态占比
status_ratio = orders["order_status"].value_counts(normalize=True)#normalize = true是转换为占比
#合并数量与占比
status_analysis = pd.DataFrame({
    "order_count": status_count,
    "order_ratio": status_ratio
})
#将占比转换为百分比
status_analysis["order_ratio"] = status_analysis["order_ratio"] *100#(1/1*100)
print(status_analysis)

              order_count  order_ratio
order_status                          
delivered           96478    97.020344
shipped              1107     1.113223
canceled              625     0.628513
unavailable           609     0.612423
invoiced              314     0.315765
processing            301     0.302692
created                 5     0.005028
approved                2     0.002011


In [4]:
#分析订单履约时间
orders [[
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].head()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
#履约流程：订单创建-订单审核-交给物流-客户签收（预计送达）

In [6]:
#指标：审核耗时；等待发货耗时；物流配送耗时
#转换日期格式
data_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in data_columns:
    orders[col] = pd.to_datetime(orders[col])#for语句，循环变量
#审核耗时
orders["approval_day"] = (
    orders["order_approved_at"]
    -
    orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400#秒化成天
orders["approval_day"]

0        0.007431
1        1.279745
2        0.011505
3        0.012419
4        0.042940
           ...   
99436    0.000000
99437    0.008090
99438    0.012188
99439    0.005486
99440    0.599282
Name: approval_day, Length: 99441, dtype: float64

In [7]:
#查看审计耗时
print(orders["approval_day"].describe())

count    99281.000000
mean         0.434129
std          1.084917
min          0.000000
25%          0.008958
50%          0.014306
75%          0.607535
max        187.882523
Name: approval_day, dtype: float64


In [8]:
#1.大部分订单审核较快；中位数0.014天约20.6分钟；2.平均值高于中位数，两者差距过大，说明存在审核耗时特别长的订单；3.最大值为187天，非常极端
#找出订单查看具体情况
slow_approval = orders[orders["approval_day"]>1]
len(slow_approval)

17419

In [9]:
#查看特征
slow_approval["approval_day"].describe()

count    17419.000000
mean         1.924375
std          1.930548
min          1.000012
25%          1.212483
50%          1.543681
75%          2.225041
max        187.882523
Name: approval_day, dtype: float64

In [10]:
#大多订单审核集中在1~2天，包括75%也就2.2天。mean和std差不多，说明存在波动
print(
slow_approval.sort_values(
    by = "approval_day",
    ascending = False,
)[[
    "order_id",
    "order_status",
    "order_approved_at",
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "approval_day"
]].head(10))

                               order_id order_status   order_approved_at  \
47552  1612081119e8f23745698ad3367cc14b  unavailable 2017-04-11 15:17:38   
62293  2e5dc86c8c4aa663549caf5e31de840d   processing 2017-04-04 10:56:48   
4541   2e7a8482f6fb09756ca50c10d7bfc047      shipped 2016-10-07 13:18:03   
4396   e5fa5a7210941f7d56d0208e4e071d35     canceled 2016-10-07 13:17:15   
43697  0a5c74ccc786ced7903270de9d6c170a  unavailable 2018-02-20 12:05:54   
96251  0a93b40850d3f4becf2f276666e01340    delivered 2018-02-20 11:51:27   
55708  f7923db0430587601c2aef15ec4b8af4    delivered 2018-02-20 12:05:54   
53475  490291524fddde2b31c2e6bec3d9e6da     canceled 2017-05-13 02:45:06   
10071  809a282bbd5dbcabb6f2f724fca862ec     canceled 2016-10-07 13:16:46   
83143  fdd647b689626410b725d1cce2ddf37c   processing 2017-12-27 14:03:00   

      order_purchase_timestamp order_delivered_carrier_date  \
47552      2016-10-05 18:06:48                          NaT   
62293      2017-02-03 00:04:49       

In [11]:
#在这10单有7单未给物流公司，2单有签收时间，只有3单有物流时间（1笔没签收时间）
#统计耗时与订单状态的关系
print(slow_approval.groupby("order_status")["approval_day"].agg
(["count", "mean","median","max"]
 ).sort_values(
    by = "count",
    ascending = False,
))

              count      mean    median         max
order_status                                       
delivered     16873  1.897396  1.540405   30.893484
shipped         204  2.034717  1.598929   32.668565
unavailable     148  3.759678  1.756777  187.882523
canceled         82  3.112636  1.647905   32.542836
processing       63  3.006444  1.591863   60.452766
invoiced         48  1.749450  1.449780    5.262037
approved          1  5.796609  5.796609    5.796609


In [12]:
#整体对照
print(orders.groupby("order_status")["approval_day"].agg
(["count", "mean","median","max"]
 ).sort_values(
    by = "count",
    ascending = False,
))

              count      mean    median         max
order_status                                       
delivered     96464  0.428199  0.014306   30.893484
shipped        1107  0.488917  0.015069   32.668565
unavailable     609  1.016199  0.018090  187.882523
canceled        484  0.611051  0.013686   32.542836
invoiced        314  0.376512  0.015370    5.262037
processing      301  0.714710  0.011863   60.452766
approved          2  2.902483  2.902483    5.796609
created           0       NaN       NaN         NaN


In [13]:
#经过对照，1.大多数的订单审核速度很快，因为deliver的平均耗时才0.4天约10小时，中位数才0.014天，约20分钟，也就是说少数极端值对平均值的影响很大
#2.unavailable值得关注，平均审核只要1天单中位数约0.02天，说明极端值的影响也很大
#3.created没有审核时间


In [19]:
#分析审核后等待交物流的时间
#等待教物流的时间=交给物流公司的时间-审核批准时间
orders["carrier_wait_day"] = (
    orders["order_delivered_carrier_date"]
    -orders["order_approved_at"]
).dt.total_seconds() / 86400
orders["carrier_wait_day"].describe()

count    97644.000000
mean         2.805038
std          3.549427
min       -171.219005
25%          0.875509
50%          1.818397
75%          3.580469
max        125.762569
Name: carrier_wait_day, dtype: float64

In [20]:
#中位数1.82天；说明订单有一半不超过约1.82天
#2.最小值为负数；代表交给物流的时间早于审核时间。
negative_wait = orders[
    orders["carrier_wait_day"]<0
]
print("负数记录数量：",len(negative_wait))
negative_wait[[
    "order_id",
    "order_status",
    "order_approved_at",
    "order_delivered_carrier_date",
    "carrier_wait_day"
]].head(10)

负数记录数量： 1359


,order_id,order_status,order_approved_at,order_delivered_carrier_date,carrier_wait_day
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-12 23:31:02,2018-06-11 14:54:00,-1.359051
64,688052146432ef8253587b930b01a06d,delivered,2018-04-24 18:25:22,2018-04-23 19:19:14,-0.962593
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-26 23:31:53,2018-07-24 12:57:00,-2.440891
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-23 12:31:53,2018-07-23 12:24:00,-0.005475
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-27 23:31:09,2018-07-24 14:03:00,-3.394549
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-05 16:33:06,2018-07-05 14:50:00,-0.071597
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-29 23:30:52,2018-07-26 14:46:00,-3.364491
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 16:52:39,2018-05-07 15:09:00,-0.071979
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 14:05:13,2018-08-16 13:27:00,-0.026539
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-05 16:31:26,2018-07-04 12:14:00,-1.178773


In [21]:
#1359条异常数据
#查看其数据统计
print(negative_wait["carrier_wait_day"].describe())

count    1359.000000
mean       -1.031333
std         4.804491
min      -171.219005
25%        -1.081528
50%        -0.715324
75%        -0.058976
max        -0.000243
Name: carrier_wait_day, dtype: float64


In [33]:
#查看比例，分析是否为少量个例还是达到了专门说明的程度
negative_count = (orders["carrier_wait_day"]<0).sum()
valid_count = orders["carrier_wait_day"].notna().sum()
negative_ratio = negative_count / valid_count *100
print("有效",valid_count)
print("异常",negative_count)
print("异常占比",round(negative_ratio,2),"%")

有效 97644
异常 1359
异常占比 1.39 %


In [38]:
#异常占比1.39%先计算正常订单吧
normal_wait = orders[orders["carrier_wait_day"] >=0]#数据库提取
print(normal_wait["carrier_wait_day"].describe())


count    96285.000000
mean         2.859186
std          3.498574
min          0.000174
25%          0.901539
50%          1.851667
75%          3.624051
max        125.762569
Name: carrier_wait_day, dtype: float64


In [39]:
#1.平均值2.86天交给物流公司
#2.中位数约1.85，说明一半订单在1.85天之前完成，那说明存在极端长时间等待记录将平均值拔高
#3.最大值125.76天，极端长时间等待记录
#以七天为观察阈值
long_wait = normal_wait[normal_wait["carrier_wait_day"]>7]

print("数量",len(long_wait))
print("平均等待时间",long_wait["carrier_wait_day"].mean())

数量 6653
平均等待时间 12.287458357340883


In [40]:
#存在长尾
#查看分布状况
long_wait["order_status"].value_counts()

order_status
delivered    6555
shipped        93
canceled        5
Name: count, dtype: int64

In [41]:
#仍存在大量收货订单，说明只是交货有点慢，并没有异常状态
#计算物流配送时间,客户实际签收时间-交给物流公司时间
orders["delivery_day"] = (
    orders["order_delivered_customer_date"]
    -orders["order_delivered_carrier_date"]
).dt.total_seconds() / 86400
print(orders["delivery_day"].describe())

count    96475.000000
mean         9.330547
std          8.760122
min        -16.096169
25%          4.099948
50%          7.099769
75%         12.029115
max        205.190972
Name: delivery_day, dtype: float64


In [45]:
#存在负值情况
#1.平均配送时间9.33天
#2.一半的订单在7.10天以上
#3.50%在7.09天，中位数低于平均数，存在配送时间长的订单

#先看负数情况

In [43]:
negative_delivery = orders[orders["delivery_day"]<0]
print("配送异常记录",len(negative_delivery))

配送异常记录 23


In [63]:
#查看这23笔订单
a=(negative_delivery["order_status"]).value_counts()
print(a)#统计字段
print(negative_delivery[[
    "order_id",
    "order_status",
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "delivery_day"
]])


order_status
delivered    23
Name: count, dtype: int64
                               order_id order_status  \
6437   a1abeb653a4d4cd1e142ccb8c82cd069    delivered   
9553   383aa8b2724fe452d9ccd9934a8c628b    delivered   
13487  cb1134f9010d242e9515ad1c78ec0c39    delivered   
14474  dceb62e8fa94b46006c9554fed743df0    delivered   
19268  5f9d46795c3126674e52becb3a1a517f    delivered   
21338  8c78d01de3a9009e23d6877a7cc9be20    delivered   
22520  b27af682321527a6349f1761eb3f360c    delivered   
25393  1cc3ae63caffff2d6c3ee3e78e074acf    delivered   
25646  e37f11cae9985ca58f0b56f268720537    delivered   
27470  fa3e37584f4fdb1ded0e0de700dfcb4e    delivered   
34939  c1e2bf2b7dd3309f2f5356c6b63968fa    delivered   
41636  b866af202be0692766081310cd4085e1    delivered   
45302  29941903985f944b0ffc49c479c1547d    delivered   
49933  76458889992169d3135b264dc13aec67    delivered   
50373  0922ee1619de7b995648e5a8407afb91    delivered   
58428  fbffbaaa30f4077e87ec6acdb835b278    delive

In [64]:
#目前已知审核后交物流时间异常结果1359条
#物流配送时间异常为23条

In [65]:
#计算正常配送时间
normal_delivery = orders[orders["delivery_day"]>=0]
print(normal_delivery["delivery_day"].describe())

count    96452.000000
mean         9.333551
std          8.758825
min          0.000000
25%          4.101525
50%          7.100312
75%         12.030443
max        205.190972
Name: delivery_day, dtype: float64


In [66]:
#1.平均配送时间为9.33天
#2.中位数只有7.10天，也说明存在长尾订单（max205天也说明了这一点）


In [77]:
#先看30天配送订单吧
long_delivery = normal_delivery[normal_delivery["delivery_day"]>30]
print("超过30天订单数",len(long_delivery))
print("平均配送时间",long_delivery["delivery_day"].mean())
print((len(long_delivery["delivery_day"] )/ len(normal_delivery["delivery_day"] )*100),"%")


超过30天订单数 2686
平均配送时间 42.73193021169438
2.7848048770372826 %


In [78]:
#分析长配送订单
long_delivery["order_status"].value_counts()

order_status
delivered    2686
Name: count, dtype: int64

In [90]:
#全是delivered订单，说明都不是异常
#分析是否真的延迟(分析实际和预计送达时间差)
long_delivery["delay_days"] = (
    long_delivery["order_delivered_customer_date"]
    -long_delivery["order_delivered_carrier_date"]
).dt.total_seconds()/86400

print(long_delivery["delay_days"].describe())

count    2686.000000
mean       42.731930
std        19.074996
min        30.003854
25%        33.199957
50%        37.286574
75%        44.428385
max       205.190972
Name: delay_days, dtype: float64


In [95]:
#配送实际时间-预计时间（>0延迟;=0刚好<0提前）
delayed_orders = long_delivery[
    long_delivery["delay_days"]>0
]
print("延迟订单数量",len(delayed_orders))
print("占比",len(delayed_orders) / len(long_delivery) * 100,"%")

延迟订单数量 2468
占比 91.88384214445271 %


In [96]:
#说明延迟订单很大一部分其实都晚于预计时间送达

In [97]:
print(delayed_orders["delay_days"].describe())

count    2468.000000
mean       19.677829
std        20.231379
min         0.335972
25%         8.873354
50%        14.883947
75%        22.823423
max       188.975081
Name: delay_days, dtype: float64


In [103]:
a = delayed_orders[delayed_orders["delay_days"]==delayed_orders["delay_days"].max()]
print(a)

                               order_id                       customer_id  \
55619  1b3190b2dfa9d789e1f14c05b647a14a  d306426abe5fca15e54b645e4462dc7b   

      order_status order_purchase_timestamp   order_approved_at  \
55619    delivered      2018-02-23 14:57:35 2018-02-23 15:16:14   

      order_delivered_carrier_date order_delivered_customer_date  \
55619          2018-02-26 18:49:07           2018-09-19 23:24:07   

      order_estimated_delivery_date  approval_day  carrier_wait_day  \
55619                    2018-03-15      0.012951          3.147836   

       delivery_day  delivery_days  delay_days  
55619    205.190972     205.190972  188.975081  


In [105]:
order_review = dataframe["olist_order_reviews_dataset"]

print(
    order_review[
        order_review["order_id"] == "1b3190b2dfa9d789e1f14c05b647a14a"
    ]
)

                            review_id                          order_id  \
700  ae83a790a3f01e1c43dd4d2863cebe0c  1b3190b2dfa9d789e1f14c05b647a14a   

     review_score review_comment_title review_comment_message  \
700             2                  NaN                    NaN   

    review_creation_date review_answer_timestamp  
700  2018-03-17 00:00:00     2018-03-17 05:08:11  


In [107]:
#提前分析发现，该用户在2018-03-17这天完成了评价，但是签收在2018-09-19中间差了大概185天，所以可能是该用户网购被自动收款了？

In [108]:
#开始分析不同月份的订单延迟分析
#目标：1.哪些鱼粉延迟最高
#2.延迟是否随月份变化？
#3.是否存在季节性或阶段性物流问题

In [115]:
#创建副本，准备分析数据
monthly_orders = orders[
    (orders["order_status"] == "delivered")&
    (orders["order_delivered_customer_date"].notna())&
    (orders["order_estimated_delivery_date"].notna())
].copy()

In [117]:
#创建延迟标记和下单月份
monthly_orders["delay_days"] = (
    monthly_orders["order_delivered_customer_date"]
    -monthly_orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

monthly_orders["is_delay"]=(
    monthly_orders["delay_days"] >0
)#延迟大于0（存在延迟）
monthly_orders["order_month"] = (
    monthly_orders["order_purchase_timestamp"].dt.to_period("M")
)

In [118]:
#按月份统计延迟率

In [122]:
monthly_delay = monthly_orders.groupby("order_month").agg(
    order_count=("order_id", "count"),
    delayed_count=("is_delay","sum"),
    delay_rate = ("is_delay","mean")
)
monthly_delay["delay_rate"] = (
    monthly_delay["delay_rate"]*100)
print(monthly_delay)

             order_count  delayed_count  delay_rate
order_month                                        
2016-09                1              1  100.000000
2016-10              265              3    1.132075
2016-12                1              0    0.000000
2017-01              750             23    3.066667
2017-02             1653             53    3.206292
2017-03             2546            142    5.577376
2017-04             2303            181    7.859314
2017-05             3545            128    3.610719
2017-06             3135            121    3.859649
2017-07             3872            133    3.434917
2017-08             4193            139    3.315049
2017-09             4150            216    5.204819
2017-10             4478            237    5.292541
2017-11             7288           1043   14.311196
2017-12             5513            462    8.380192
2018-01             7069            464    6.563870
2018-02             6555           1048   15.987796
2018-03     

In [124]:
#可以看出2017-1到-10中间是比较稳定的，趋于3%-8%之间
#2017—11则到了14出现高峰期
#2018-02和-03包括2018-8都出现高峰期，分别是15.99；21.36；10.39
#得出结论是存在明显波动

In [127]:
#找出延迟高峰月份的订单量
monthly_delay[[
"order_count",
"delayed_count",
"delay_rate",
]].sort_values(
    by ="delay_rate",
    ascending = False,
).head(10)

,order_count,delayed_count,delay_rate
order_month,,,
2016-09,1,1,100.000000
2018-03,7003,1496,21.362273
2018-02,6555,1048,15.987796
2017-11,7288,1043,14.311196
2018-08,6351,660,10.392064
2017-12,5513,462,8.380192
2018-05,6749,556,8.238258
2017-04,2303,181,7.859314
2018-01,7069,464,6.563870


In [134]:
print(
    monthly_delay.sort_values(
        by="delay_rate",
        ascending=False
    )
)

             order_count  delayed_count  delay_rate
order_month                                        
2016-09                1              1  100.000000
2018-03             7003           1496   21.362273
2018-02             6555           1048   15.987796
2017-11             7288           1043   14.311196
2018-08             6351            660   10.392064
2017-12             5513            462    8.380192
2018-05             6749            556    8.238258
2017-04             2303            181    7.859314
2018-01             7069            464    6.563870
2017-03             2546            142    5.577376
2018-04             6798            361    5.310385
2017-10             4478            237    5.292541
2017-09             4150            216    5.204819
2018-07             6156            276    4.483431
2017-06             3135            121    3.859649
2017-05             3545            128    3.610719
2017-07             3872            133    3.434917
2017-08     

In [131]:
#无法解释订单越高延迟越高，或许会增加物流压力但仍然是无法解释演示率的变化

In [132]:
#计算各月份的平均延迟天数(包括提前到的)
monthly_delay_days = monthly_orders.groupby(
    "order_month"
)["delay_days"].mean()

monthly_delay_days

order_month
2016-09    36.324745
2016-10   -36.076073
2016-12   -21.336991
2017-01   -26.861788
2017-02   -18.680104
2017-03   -11.781202
2017-04   -12.431898
2017-05   -12.962421
2017-06   -12.010292
2017-07   -11.724584
2017-08   -12.333478
2017-09   -10.586360
2017-10   -11.162836
2017-11    -7.399620
2017-12   -12.286421
2018-01   -12.221993
2018-02    -7.583858
2018-03    -5.731657
2018-04   -12.180305
2018-05   -11.472459
2018-06   -18.533524
2018-07   -10.731490
2018-08    -7.453550
Freq: M, Name: delay_days, dtype: float64

In [138]:
#均为负数，也就是说整体的延迟率并不高
#计算平均每个月的平均延迟天数
monthly_late_days = (
    monthly_orders[monthly_orders["delay_days"]>0]#延迟天数大于0
    .groupby("order_month")["delay_days"]
    .mean()
)
monthly_late_days.sort_values(
    ascending = False
)

order_month
2016-09    36.324745
2017-03    21.322515
2017-01    19.848682
2017-02    19.728950
2018-06    12.139928
2017-06    11.874735
2018-02    11.821123
2018-01    11.022545
2017-05    10.932780
2017-04    10.814364
2017-11    10.391889
2017-07    10.221371
2018-03     9.580825
2018-04     9.512524
2017-12     9.357434
2017-09     8.649238
2017-08     8.426372
2017-10     8.005391
2018-07     7.003381
2016-10     6.944313
2018-05     6.360927
2018-08     3.282122
Freq: M, Name: delay_days, dtype: float64

In [ ]:
#延迟率和延迟程度
#2018-3；延续范围最大，达到23.36；但平均晚到9.58天
#2018-6；延迟范围较低（倒数第三），1.36，但平均晚到12.14天
#2018-8；延迟率较高，10.39，但是平均晚到最低3.2
#所以不能用延迟率评价物流表现，还需要考虑延迟严重程度